# Train VAE on Colab

Runtime > Change runtime type > **T4 GPU** before running.
Edit `USERNAME` in cell 2, then run the cells in order.

## 1. GPU check and Google Drive

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive')

## 2. Clone the repo and install dependencies

In [ ]:
import os
%cd /content
if not os.path.exists("vae-ddpm-from-scratch"):
    !git clone https://github.com/USERNAME/vae-ddpm-from-scratch.git
%cd /content/vae-ddpm-from-scratch
!git pull
!pip install -q torch-fidelity pyyaml

## 3. Train (resumes automatically from the last checkpoint on Drive)

In [ ]:
import sys, os, yaml
sys.path.append(os.getcwd())
from src.vae.train import train

OUT_DIR = "/content/drive/MyDrive/vae_ddpm/vae"

with open("configs/vae.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

# override any value here instead of editing the yaml
cfg["output_dir"] = OUT_DIR
cfg["epochs"] = 50

train(cfg, resume=True)

## 4. Look at the results

In [ ]:
import glob, json
import matplotlib.pyplot as plt
from IPython.display import Image, display

# latest generated samples
files = sorted(glob.glob(f"{OUT_DIR}/samples/epoch_*.png"),
               key=lambda p: int(p.split("_")[-1].split(".")[0]))
display(Image(files[-1]))

# loss curves
with open(f"{OUT_DIR}/plots/history.json", encoding="utf-8") as f:
    h = json.load(f)
hist = h["history"]
epochs = [e["epoch"] for e in hist]

plt.figure(figsize=(8, 4))
plt.plot(epochs, [e["loss"] for e in hist], label="total")
plt.plot(epochs, [e["recon"] for e in hist], label="reconstruction")
plt.plot(epochs, [e["kl"] for e in hist], label="KL")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.grid(alpha=0.3)
plt.savefig(f"{OUT_DIR}/plots/loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Total training time: {h['train_time_sec'] / 60:.1f} min")